# Quantization

## Learning Objectives
1. Understand INT8 quantization: how scale and zero_point map float32 tensors to 8-bit integers.
2. Apply dynamic and static post-training quantization with `torch.quantization` and handle OOM.
3. Quantize BERT with PTQ to achieve ~4x model-size reduction while measuring accuracy impact.
4. Compare INT8 vs INT4 trade-offs in terms of accuracy degradation, size reduction, and latency.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import copy

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: INT8 Quantization — Scale and Zero-Point (NumPy)

In [ ]:
# Demonstrate affine (asymmetric) quantization: maps fp32 range [r_min, r_max]
# to INT8 range [-128, 127] using scale and zero_point.

def compute_scale_zero_point(
    r_min: float, r_max: float, n_bits: int = 8
) -> tuple:
    """Compute quantization parameters for affine mapping.
    
    q = clamp(round(r / scale + zero_point), q_min, q_max)
    r_approx = (q - zero_point) * scale
    """
    q_min = -(2 ** (n_bits - 1))
    q_max = 2 ** (n_bits - 1) - 1
    scale = (r_max - r_min) / (q_max - q_min)
    zero_point = int(round(q_min - r_min / scale))
    zero_point = np.clip(zero_point, q_min, q_max)
    return float(scale), int(zero_point), q_min, q_max


def quantize(tensor: np.ndarray, scale: float, zero_point: int,
             q_min: int, q_max: int) -> np.ndarray:
    """Map fp32 tensor to integers."""
    return np.clip(np.round(tensor / scale + zero_point), q_min, q_max).astype(np.int8)


def dequantize(q_tensor: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    """Reconstruct fp32 approximation from integers."""
    return (q_tensor.astype(np.float32) - zero_point) * scale


# Test on a simulated weight vector
rng = np.random.default_rng(42)
weights = rng.normal(0, 0.1, 512).astype(np.float32)

for n_bits in [8, 4]:
    scale, zp, qmin, qmax = compute_scale_zero_point(
        weights.min(), weights.max(), n_bits
    )
    q = quantize(weights, scale, zp, qmin, qmax)
    deq = dequantize(q, scale, zp)
    mse = np.mean((weights - deq) ** 2)
    size_ratio = 32 / n_bits
    print(f"INT{n_bits:2d} | scale={scale:.5f} zero_point={zp:4d} | "
          f"MSE={mse:.6f} | size_ratio={size_ratio:.1f}x")

## Level 2: Dynamic and Static Post-Training Quantization (PyTorch)

In [ ]:
# Dynamic: scales computed at runtime per-activation (easy, no calibration needed).
# Static: scales fixed from calibration data (faster inference, needs calibration).

import os

class SimpleTextModel(nn.Module):
    """Small FC model that mimics an embedding + classifier pipeline."""
    def __init__(self, vocab: int = 1000, embed: int = 64, hidden: int = 128, n_classes: int = 10):
        super().__init__()
        self.embed = nn.EmbeddingBag(vocab, embed, mode='mean')
        self.fc1 = nn.Linear(embed, hidden)
        self.fc2 = nn.Linear(hidden, n_classes)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.relu(self.fc1(self.embed(x))))


def model_size_mb(model: nn.Module) -> float:
    """Estimate model size in MB from parameter count."""
    total_params = sum(p.numel() * p.element_size() for p in model.parameters())
    return total_params / 1e6


# Create fp32 model on CPU (quantization works on CPU)
model_fp32 = SimpleTextModel().cpu()
print(f"FP32 model size: {model_size_mb(model_fp32):.3f} MB")

# --- Dynamic Quantization (weights INT8, activations computed dynamically) ---
try:
    model_dynamic = copy.deepcopy(model_fp32)
    model_dynamic = torch.quantization.quantize_dynamic(
        model_dynamic,
        {nn.Linear},  # Only quantize linear layers
        dtype=torch.qint8,
    )
    # Size estimate is approximate since quantized weights have custom repr
    print(f"Dynamic INT8 model quantized successfully")
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print("OOM during quantization — model too large for available RAM")
    else:
        raise

# --- Static Quantization (requires calibration + QConfig) ---
class QuantizableModel(nn.Module):
    """Wrapped model with QuantStub/DeQuantStub for static quantization."""
    def __init__(self):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.quant(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return self.dequant(x)


model_static = QuantizableModel().cpu()
model_static.eval()
model_static.qconfig = torch.quantization.get_default_qconfig('fbgemm')
torch.quantization.prepare(model_static, inplace=True)

# Calibration pass (representative data subset)
calib_data = torch.randn(200, 64)
with torch.no_grad():
    for i in range(0, 200, 32):
        model_static(calib_data[i:i+32])

torch.quantization.convert(model_static, inplace=True)
print(f"Static INT8 model converted successfully")

# Accuracy check on synthetic data
test_in = torch.randn(64, 64)
with torch.no_grad():
    out_q = model_static(test_in)
print(f"Static INT8 output shape: {out_q.shape}  max_logit={out_q.max().item():.3f}")

## Real-World Example 1: Post-Training Quantization of BERT (4x Size Reduction)

In [ ]:
# Simulate PTQ on a BERT-scale linear layer stack and measure size reduction.
# In production: load bert-base-uncased, apply quantize_dynamic, benchmark.

import time

class BERTLikeEncoder(nn.Module):
    """Mimics BERT encoder blocks with 12 x (768->3072->768) linear layers."""
    def __init__(self, n_layers: int = 12, d_model: int = 768, ffn_dim: int = 3072):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, ffn_dim),
                nn.ReLU(),
                nn.Linear(ffn_dim, d_model),
            ) for _ in range(n_layers)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = x + layer(x)  # Residual connection
        return x


bert_fp32 = BERTLikeEncoder().cpu().eval()

def param_count_mb(m: nn.Module) -> float:
    return sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6

size_fp32 = param_count_mb(bert_fp32)
print(f"FP32 BERT-like encoder: {size_fp32:.1f} MB")

# Apply dynamic INT8 quantization (most practical for BERT inference)
bert_int8 = torch.quantization.quantize_dynamic(
    copy.deepcopy(bert_fp32), {nn.Linear}, dtype=torch.qint8
)

# Benchmark latency
dummy = torch.randn(1, 128, 768)  # [batch, seq_len, hidden]
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(20):
        bert_fp32(dummy)
    lat_fp32 = (time.perf_counter() - t0) / 20 * 1000

    t0 = time.perf_counter()
    for _ in range(20):
        bert_int8(dummy)
    lat_int8 = (time.perf_counter() - t0) / 20 * 1000

print(f"FP32 latency: {lat_fp32:.1f}ms")
print(f"INT8 latency: {lat_int8:.1f}ms  ({lat_fp32/lat_int8:.1f}x speedup)")
print(f"Size reduction: ~4x (fp32 -> int8 reduces each param from 4B to 1B)")

## Real-World Example 2: Quantization-Aware Training (QAT) to Recover Accuracy

In [ ]:
# QAT inserts fake-quantization ops during training so the model learns
# to tolerate quantization noise — recovering accuracy lost in PTQ.

from torch.utils.data import DataLoader, TensorDataset

class SmallQATModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.fc1 = nn.Linear(32, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)
        self.relu = nn.ReLU()
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.quant(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.dequant(self.fc3(x))
        return x


# Synthetic binary classification data
X_train = torch.randn(800, 32)
y_train = (X_train[:, 0] > 0).long()
X_val = torch.randn(200, 32)
y_val = (X_val[:, 0] > 0).long()
loader_tr = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

def eval_acc(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X).argmax(1)
    return (preds == y).float().mean().item()

# Step 1: Pre-train in fp32
model_qat = SmallQATModel().cpu()
opt = torch.optim.Adam(model_qat.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
for _ in range(15):
    model_qat.train()
    for xb, yb in loader_tr:
        opt.zero_grad()
        crit(model_qat(xb), yb).backward()
        opt.step()
acc_fp32 = eval_acc(model_qat, X_val, y_val)
print(f"FP32 pre-train val acc: {acc_fp32:.3f}")

# Step 2: PTQ to see accuracy drop
model_ptq = copy.deepcopy(model_qat)
model_ptq.eval()
model_ptq.qconfig = torch.quantization.get_default_qconfig('fbgemm')
torch.quantization.prepare(model_ptq, inplace=True)
with torch.no_grad():
    model_ptq(X_val)  # Calibrate
torch.quantization.convert(model_ptq, inplace=True)
acc_ptq = eval_acc(model_ptq, X_val, y_val)
print(f"PTQ INT8 val acc:       {acc_ptq:.3f}")

# Step 3: QAT — fine-tune with fake quant ops
model_qat_finetuned = copy.deepcopy(model_qat)
model_qat_finetuned.train()
model_qat_finetuned.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
torch.quantization.prepare_qat(model_qat_finetuned, inplace=True)
opt_qat = torch.optim.Adam(model_qat_finetuned.parameters(), lr=1e-4)
for _ in range(10):
    for xb, yb in loader_tr:
        opt_qat.zero_grad()
        crit(model_qat_finetuned(xb), yb).backward()
        opt_qat.step()
model_qat_finetuned.eval()
torch.quantization.convert(model_qat_finetuned, inplace=True)
acc_qat = eval_acc(model_qat_finetuned, X_val, y_val)
print(f"QAT INT8 val acc:       {acc_qat:.3f}")
print(f"\nQAT recovered {acc_qat - acc_ptq:.3f} accuracy vs PTQ")

## Real-World Example 3: INT8 vs INT4 Trade-off Analysis

In [ ]:
# Compare INT8 vs INT4 quantization: reconstruction error, size savings,
# and practical accuracy impact across weight distributions.

import numpy as np

def quantization_error(n_bits: int, distribution: str = 'normal') -> dict:
    """Compute quantization statistics for a simulated weight tensor."""
    rng = np.random.default_rng(42)
    if distribution == 'normal':
        w = rng.normal(0, 0.02, 100_000).astype(np.float32)
    elif distribution == 'laplace':
        # LoRA / small delta weights often Laplace-distributed
        w = rng.laplace(0, 0.01, 100_000).astype(np.float32)
    else:
        w = rng.uniform(-0.1, 0.1, 100_000).astype(np.float32)

    scale, zp, qmin, qmax = compute_scale_zero_point(w.min(), w.max(), n_bits)
    q = quantize(w, scale, zp, qmin, qmax)
    deq = dequantize(q, scale, zp)
    mse = float(np.mean((w - deq) ** 2))
    snr_db = float(10 * np.log10(np.mean(w**2) / (mse + 1e-15)))
    return {'mse': mse, 'snr_db': snr_db, 'size_reduction': f"{32/n_bits:.0f}x"}


print(f"{'Bits':>5} {'Distribution':>12} {'MSE':>12} {'SNR (dB)':>10} {'Size':>6}")
print("-" * 55)
for bits in [8, 4]:
    for dist in ['normal', 'laplace', 'uniform']:
        stats = quantization_error(bits, dist)
        print(f"INT{bits:1d}  {dist:>12}  {stats['mse']:12.8f}  {stats['snr_db']:10.2f}  "
              f"{stats['size_reduction']:>6}")

print("\nPractical guidance:")
print("  INT8: <0.5% accuracy loss on most models — safe default for production.")
print("  INT4: 1-3% accuracy loss — requires QAT or GPTQ for LLMs to stay within budget.")
print("  Use INT4 when memory is the bottleneck (edge devices, large LLMs in 8GB VRAM).")